In [1]:
import msprime
import numpy as np
import pandas as pd
import math
from collections import defaultdict

In [5]:
dem = msprime.Demography()
dem.add_population(name="A", initial_size=4000)
ts = msprime.sim_ancestry(samples={"A": 20}, demography=dem, sequence_length=3e8, recombination_rate=1e-8, random_seed=1, model = 'smc') 
mutated_ts = msprime.sim_mutations(ts, rate=1e-8, random_seed=1)


In [7]:
mutated_ts.num_trees

202092

In [8]:
def all_ibd_segments(ts):
    """
    Manually extracts IBD segments by iterating trees, 
    but MERGES adjacent trees if the TMRCA for the pair is unchanged.
    """
    # Initialize a matrix to store current segment starts for every pair
    n = ts.num_samples
    current_mrca = np.zeros((n, n)) - 1
    current_start = np.zeros((n, n))
    
    # Store completed segments: segments[u][v] = [len1, len2, ...]
    segments = defaultdict(lambda: defaultdict(list))
    
    # Iterate continuously through the genome
    for tree in ts.trees():
        interval_end = tree.interval.right
        
        # This is efficient for small n, slow for large n
        # For every pair, check their MRCA time/node
        for i in range(n):
            for j in range(i + 1, n):
                mrca_node = tree.mrca(i, j)
                
                # If this is the first tree, initialize
                if current_mrca[i, j] == -1:
                    current_mrca[i, j] = mrca_node
                    current_start[i, j] = tree.interval.left
                
                # If MRCA changed, the segment ended. Record it.
                elif current_mrca[i, j] != mrca_node:
                    # Calculate length (fraction of genome or bp)
                    # Here we store fraction to match your 'l' definition
                    seg_len = (tree.interval.left - current_start[i, j]) / ts.sequence_length
                    segments[i][j].append(seg_len)
                    
                    # Start new segment
                    current_mrca[i, j] = mrca_node
                    current_start[i, j] = tree.interval.left
                    
    # Flush the final segments at the end of the chromosome
    for i in range(n):
        for j in range(i + 1, n):
            seg_len = (ts.sequence_length - current_start[i, j]) / ts.sequence_length
            segments[i][j].append(seg_len)
            
    return segments

In [10]:
M = all_ibd_segments(mutated_ts)

In [20]:
len(M)

39

In [29]:
bins = [[0.8,1],[1,300]]
total_frac = {bin[0]: 0 for bin in bins}
for bin in bins:
    for i in range(40):
        for j in range(i+1,40):
            seg = M[i][j]
            total_frac[bin[0]] += sum([s for s in seg if bin[0] <= s * 300 < bin[1]])



In [37]:
total_frac[bins[0][0]]/(20*39)

np.float64(0.002988398029914533)

In [38]:
total_frac[bins[1][0]]/(20*39)

np.float64(0.011717027675213666)

In [50]:
def calculate_ibd_fractions(ts, bins, cm_per_unit=1e-6, num_bootstraps=1000):    

    
    sample_nodes = ts.samples()
    node_to_pop = ts.nodes_population[sample_nodes]
    pop_ids = np.unique(node_to_pop)
    num_pops = len(pop_ids)
    

    pop_samples = defaultdict(list)
    for u in sample_nodes:
        pop_samples[node_to_pop[u]].append(u)

    results = {b_i: defaultdict(lambda: defaultdict(float)) 
               for b_i in range(len(bins))}
    
    genome_length = ts.sequence_length * cm_per_unit
    
    # Filter tiny segments
    min_bin_val = min(b[0] for b in bins)
    min_span_ts_units = min_bin_val / cm_per_unit
    
    ibd_iter = ts.ibd_segments(
        store_pairs=True, 
        store_segments=True,
        min_span=0
    )

    print(f"Iterating IBD segments (min_span={min_span_ts_units:.2f})...")
    
    for (u, v), segments in ibd_iter.items(): 
        p_u = ts.nodes_population[u]
        p_v = ts.nodes_population[v]
        p_i, p_j = sorted((p_u, p_v))
        
        # Unique identifier for this specific pair of individuals
        pair_key = tuple(sorted((u, v)))
        
        for seg in segments:
            seg_len = (seg.right - seg.left) * cm_per_unit
            
            for b_i, (min_len, max_len) in enumerate(bins):
                if min_len <= seg_len < max_len:
                    # FIX 3: Store fraction for the pair using tuple key (p_i, p_j)
                    # This matches how the bootstrap loop tries to retrieve it later.
                    results[b_i][(p_i, p_j)][pair_key] += (seg_len / genome_length)
                    break 
    
    final_mean_matrix = {}
    final_var_matrix = {}

    for b_i in results:
        mean_matrix = np.zeros((num_pops, num_pops))
        var_matrix = np.zeros((num_pops, num_pops))

        for i in range(num_pops):
            for j in range(i, num_pops):
                # Calculate total theoretical pairs (N)
                # Now this works because pop_samples[i] is a list
                if i == j:
                    n = len(pop_samples[i]) #
                    num_pairs = n * (n - 1) // 2
                else:
                    num_pairs = len(pop_samples[i]) * len(pop_samples[j])
                
                if num_pairs == 0:
                    continue

                # Retrieve observed non-zero fractions
                # This works now because we stored data with key (i, j)
                observed_dict = results[b_i].get((i, j), {})
                observed_values = np.array(list(observed_dict.values()))
                
                # The rest are zeros
                count_zeros = num_pairs - len(observed_values)
                
                # Construct the full population of pairs
                full_population = np.concatenate([
                    observed_values, 
                    np.zeros(count_zeros)
                ])

                # A. Original Mean
                original_mean = np.mean(full_population)
                
                # B. Bootstrap Variance
                if num_bootstraps > 0:
                    boot_samples = np.random.choice(full_population, size=(num_bootstraps, num_pairs), replace=True)
                    boot_means = np.mean(boot_samples, axis=1)
                    boot_var = np.var(boot_means)
                else:
                    boot_var = 0.0

                # Fill Matrices
                mean_matrix[i, j] = mean_matrix[j, i] = original_mean
                var_matrix[i, j] = var_matrix[j, i] = boot_var

        final_mean_matrix[b_i] = mean_matrix
        final_var_matrix[b_i] = var_matrix

    return final_mean_matrix, final_var_matrix

In [51]:
a,b = calculate_ibd_fractions(mutated_ts, bins)

Iterating IBD segments (min_span=800000.00)...


In [53]:
a

{0: array([[0.00186062]]), 1: array([[0.00869987]])}

In [47]:
2771152958.0/(20*39)/(3e8)

0.011842534008547009